# Week 5 Post-Class Exercise: Iris Clustering & PCA - SOLUTIONS

**Scaffolding:** Complete solutions  
**Dataset:** Iris (150 samples, 4 features)  

---

## Part 1: Load & Explore Data

In [ ]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Set style for better-looking plots
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

In [ ]:
# Load the iris dataset
iris = load_iris()
X = iris.data
feature_names = iris.feature_names

# Create a DataFrame for easier analysis
df = pd.DataFrame(X, columns=feature_names)

print(f"Dataset shape: {X.shape}")
print(f"\nFeature names: {feature_names}")
df.head()

In [ ]:
# Check basic statistics
df.describe()

### Answers

**Q1: Do features have similar scales?**
- **Answer:** No. Petal length ranges from 1-6.9 cm, while sepal width ranges from 2-4.4 cm. Features have different scales.

**Q2: Why does this matter for K-Means?**
- **Answer:** K-Means uses Euclidean distance. Features with larger scales will dominate the distance calculation, biasing the clustering. We must scale features so each contributes equally.

---

## Part 2: Feature Scaling

In [ ]:
# Scale the features using StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

print("✅ Features scaled")
print(f"Mean of scaled features: {X_scaled.mean(axis=0).round(2)}")
print(f"Std of scaled features: {X_scaled.std(axis=0).round(2)}")

---

## Part 3: Choose Optimal K

In [ ]:
# Implement elbow method
inertias = []
K_range = range(1, 11)

for k in K_range:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    kmeans.fit(X_scaled)
    inertias.append(kmeans.inertia_)

# Plot
plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, marker='o', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Inertia (Within-Cluster Sum of Squares)', fontsize=12)
plt.title('Elbow Method for Optimal K', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.axvline(x=3, color='red', linestyle='--', alpha=0.5, label='Elbow at K=3')
plt.legend()
plt.tight_layout()
plt.show()

print("✅ Elbow appears at K=3")

In [ ]:
# Implement silhouette analysis
silhouette_scores = []
K_range_sil = range(2, 9)

print("Silhouette Scores:")
print("-" * 40)

for k in K_range_sil:
    kmeans = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = kmeans.fit_predict(X_scaled)
    score = silhouette_score(X_scaled, labels)
    silhouette_scores.append(score)
    print(f"K={k}: Silhouette Score = {score:.3f}")

# Plot
plt.figure(figsize=(8, 5))
plt.plot(K_range_sil, silhouette_scores, marker='o', color='orange', linewidth=2, markersize=8)
plt.xlabel('Number of Clusters (K)', fontsize=12)
plt.ylabel('Silhouette Score', fontsize=12)
plt.title('Silhouette Analysis for Optimal K', fontsize=14, fontweight='bold')
plt.axhline(y=0.5, color='gray', linestyle='--', label='Good threshold (0.5)', alpha=0.5)
plt.axvline(x=2, color='green', linestyle='--', alpha=0.5, label='Highest at K=2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"\n✅ Highest silhouette score at K={K_range_sil[np.argmax(silhouette_scores)]}")

### Answers: Decide on K

**Q3: Where is the elbow?**
- **Answer:** K=3. The inertia decrease slows significantly after K=3.

**Q4: Which K has the highest silhouette score?**
- **Answer:** K=2 (score ~0.68), but K=3 also has a strong score (~0.55).

**Q5: What K will you choose and why?**
- **Answer:** K=3. While K=2 has the highest silhouette score, the elbow method clearly points to K=3. Additionally, the Iris dataset has 3 species, making K=3 a reasonable biological choice. Both methods support K=3 as a strong candidate.

---

## Part 4: Fit K-Means

In [ ]:
# Fit K-Means with chosen K
K_chosen = 3

kmeans = KMeans(n_clusters=K_chosen, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_scaled)

# Add to DataFrame
df['cluster'] = clusters

print(f"✅ K-Means fitted with K={K_chosen}")
print(f"\nCluster distribution:")
print(df['cluster'].value_counts().sort_index())

---

## Part 5: Interpret Clusters

In [ ]:
# Calculate mean values for each cluster
cluster_means = df.groupby('cluster').mean()

print("Cluster Characteristics (Mean Values):")
print("=" * 60)
cluster_means

In [ ]:
# Visualize cluster characteristics with a heatmap
plt.figure(figsize=(10, 6))
sns.heatmap(cluster_means.T, annot=True, fmt='.2f', cmap='coolwarm', center=cluster_means.values.mean(), 
            linewidths=1, cbar_kws={'label': 'Mean Value (cm)'})
plt.title('Cluster Characteristics Heatmap', fontsize=14, fontweight='bold')
plt.xlabel('Cluster', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.tight_layout()
plt.show()

### Cluster Naming

**Cluster 0:**
- **Defining features:** Small petals (petal length ~1.5cm, petal width ~0.2cm), moderate sepals
- **Proposed name:** "Small-Petaled Iris" or "Compact Flowers"
- **Description:** These irises have distinctly small petals compared to other groups. They likely represent a species with compact flower morphology.

**Cluster 1:**
- **Defining features:** Large petals (petal length ~5.6cm, petal width ~2.0cm), large sepals (sepal length ~6.9cm)
- **Proposed name:** "Large-Flowered Iris" or "Showy Blooms"
- **Description:** These irises have the largest flowers overall, with both large petals and sepals. They are likely a species bred for visual impact.

**Cluster 2:**
- **Defining features:** Medium-sized petals (petal length ~4.3cm, petal width ~1.3cm), moderate sepals
- **Proposed name:** "Medium-Flowered Iris" or "Balanced Blooms"
- **Description:** These irises fall between the small and large groups, with moderate dimensions across all features.

---

## Part 6: PCA Visualization

In [ ]:
# Apply PCA to reduce to 2 components
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

print(f"PC1 variance explained: {pca.explained_variance_ratio_[0]:.1%}")
print(f"PC2 variance explained: {pca.explained_variance_ratio_[1]:.1%}")
print(f"Total variance explained: {pca.explained_variance_ratio_.sum():.1%}")

In [ ]:
# Visualize clusters in PCA space
plt.figure(figsize=(10, 7))
scatter = plt.scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis', 
                     s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=12)
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=12)
plt.title('Iris Clusters Visualized in PCA Space', fontsize=14, fontweight='bold')
plt.colorbar(scatter, label='Cluster')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

### Answers: Analyze PCA Results

**Q6: What % of total variance is explained by PC1 + PC2?**
- **Answer:** ~95.8% (PC1: 72.9%, PC2: 22.9%)

**Q7: Are clusters well-separated in PCA space?**
- **Answer:** Yes! Cluster 0 (small-petaled) is completely separated on the left. Clusters 1 and 2 (medium and large) have some overlap but are mostly distinct.

**Q8: Is this % variance "good enough" for visualization?**
- **Answer:** Absolutely! 95.8% means we're retaining almost all information when projecting from 4D to 2D. This is excellent for visualization - we can trust that the 2D plot accurately represents the true cluster structure.

---

## Part 7: BONUS - Clustering AFTER PCA

In [ ]:
# Fit K-Means on PCA space (2D)
kmeans_pca = KMeans(n_clusters=K_chosen, random_state=42, n_init=10)
clusters_pca = kmeans_pca.fit_predict(X_pca)

# Add to DataFrame
df['cluster_pca'] = clusters_pca

print("Cluster distribution (PCA-based):")
print(df['cluster_pca'].value_counts().sort_index())

In [ ]:
# Compare original clusters vs PCA-based clusters
comparison = pd.crosstab(df['cluster'], df['cluster_pca'], 
                        rownames=['Original Space'], colnames=['PCA Space'])

print("\nComparison: Original Space vs PCA Space Clustering")
print("=" * 50)
print(comparison)

# Calculate agreement percentage
agreement = (df['cluster'] == df['cluster_pca']).sum() / len(df) * 100
print(f"\nAgreement: {agreement:.1f}% of samples assigned to same cluster")

In [ ]:
# Visualize side-by-side comparison
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Clustering in original space (visualized via PCA)
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis',
                          s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
axes[0].set_title('K-Means on Original 4D Space', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Cluster')

# Plot 2: Clustering in PCA space
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters_pca, cmap='viridis',
                          s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
axes[1].set_title('K-Means on PCA 2D Space', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
plt.colorbar(scatter2, ax=axes[1], label='Cluster')

plt.tight_layout()
plt.show()

### BONUS: Reflection Answers

**Q9: Do clustering results differ when using PCA space vs original space?**
- **Answer:** Very slightly! Agreement is typically >95%. Because PCA retains 95.8% of variance, clustering in 2D PCA space is nearly identical to clustering in the original 4D space. The small differences come from the 4.2% of variance we discarded.

**Q10: Which approach do you prefer and why?**
- **Answer:** For Iris, clustering in original 4D space is preferable because:
  1. We retain ALL information (no variance loss)
  2. Results are nearly identical to PCA-based clustering anyway
  3. We can still USE PCA for visualization afterward
  
  However, for high-dimensional data (e.g., 100+ features), clustering AFTER PCA can be beneficial for:
  - Noise reduction
  - Computational efficiency
  - Avoiding curse of dimensionality

---

## Part 8: Compare with True Labels (BONUS)

In [ ]:
# The Iris dataset has true species labels
true_labels = iris.target
target_names = iris.target_names

df['true_species'] = true_labels

# Compare clusters to true species
comparison = pd.crosstab(df['cluster'], df['true_species'])
comparison.columns = target_names

print("\nCluster vs True Species:")
print("=" * 60)
print(comparison)
print("\n" + "=" * 60)

# Calculate purity for each cluster
print("\nCluster Purity Analysis:")
for cluster_id in range(K_chosen):
    cluster_data = df[df['cluster'] == cluster_id]
    most_common_species = cluster_data['true_species'].mode()[0]
    purity = (cluster_data['true_species'] == most_common_species).sum() / len(cluster_data) * 100
    species_name = target_names[most_common_species]
    print(f"Cluster {cluster_id}: {purity:.1f}% {species_name}")

In [ ]:
# Visualize: clusters vs true labels side-by-side
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Your clusters
scatter1 = axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=clusters, cmap='viridis',
                          s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[0].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
axes[0].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
axes[0].set_title('Your K-Means Clusters', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)
plt.colorbar(scatter1, ax=axes[0], label='Cluster', ticks=[0, 1, 2])

# Plot 2: True species
scatter2 = axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=true_labels, cmap='viridis',
                          s=80, alpha=0.7, edgecolors='black', linewidth=0.5)
axes[1].set_xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} variance)', fontsize=11)
axes[1].set_ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} variance)', fontsize=11)
axes[1].set_title('True Species Labels', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)
cbar = plt.colorbar(scatter2, ax=axes[1], label='Species', ticks=[0, 1, 2])
cbar.ax.set_yticklabels(target_names)

plt.tight_layout()
plt.show()

### Final Reflection Answers

**Q11: How well do your clusters match the true species?**
- **Answer:** Very well! Cluster 0 perfectly captures setosa (100% purity). Clusters 1 and 2 mostly separate versicolor and virginica, though there's some overlap. Overall clustering accuracy is ~90%.

**Q12: If there are differences, what might explain them?**
- **Answer:** K-Means finds clusters based on Euclidean distance in feature space - it groups flowers with similar measurements. Species, however, are defined by biological taxonomy, not just measurements. Versicolor and virginica have overlapping measurements (especially petal dimensions), so K-Means sometimes groups them together even though they're different species. This is expected: **statistical similarity ≠ biological category**.

**Q13: In a real-world scenario WITHOUT true labels, how would you validate your clusters?**
- **Answer:**
  1. **Silhouette scores:** Measure cluster cohesion and separation
  2. **Domain expert review:** Have botanists examine cluster characteristics and assess if they make botanical sense
  3. **Visualization:** Use PCA plots to visually confirm clusters are well-separated
  4. **Stability analysis:** Run K-Means multiple times with different initializations; stable clusters are more trustworthy
  5. **Business validation:** If clustering is for marketing/inventory, test if the segments lead to better business outcomes

---

## 🎯 Key Insights from Solutions

**What you learned:**

1. ✅ **Feature scaling is critical:** K-Means is sensitive to scale differences
2. ✅ **Multiple methods for choosing K:** Combine elbow + silhouette + domain knowledge
3. ✅ **PCA retains structure:** 95.8% variance means 2D visualization is highly reliable
4. ✅ **Clustering in original vs PCA space:** For Iris, results are nearly identical because PCA retains so much variance
5. ✅ **Unsupervised ≠ Ground truth:** Clusters don't perfectly match species - and that's OK! K-Means finds statistical patterns, not biological categories.
6. ✅ **Interpretation matters:** Naming clusters based on feature analysis helps communicate findings

**Common mistakes to avoid:**
- ❌ Forgetting to scale features before K-Means
- ❌ Choosing K based ONLY on elbow or ONLY on silhouette (use both!)
- ❌ Over-interpreting low variance PCA components
- ❌ Expecting unsupervised clusters to match labeled categories perfectly

---

## 📚 Next Steps

1. **Compare your work:** How did your answers differ from these solutions?
2. **Experiment:** Try K=2 or K=4 and see how interpretations change
3. **Bonus notebook:** Complete the PCA preprocessing notebook
4. **Reflect:** Fill out `Week5_Self_Assessment.md`

---

**Excellent work completing Week 5! You're ready for Week 6 - Neural Networks! 🚀**